# 00 - Shared Drive data

The big Zenodo files already live in the shared Drive folder `Cadenza Challenge`, so nobody re-downloads them from Zenodo.

**One-time (per person):** open the shared folder in Drive -> right-click -> *Organize* -> *Add shortcut* -> **My Drive** (keep the name `Cadenza Challenge`).

These files must sit directly in that folder (not in a subfolder):
`metadata.zip`, `EnsembleSet_Mix_1.zip`, `CadenzaWoodwind.zip`, `cadenza_cad2_task2_eval.v1_1.tar.gz`

- **Part A** - copy the zips from Drive to Colab's local disk (fast), check md5, and build the dataset.
- **Part B** (optional) - copy and extract the eval set used by notebook 02.

Run `01_setup.ipynb` first (Clarity cloned + installed). Needs ~30 GB free on the Colab disk.
No Google Drive app is needed on your laptop - Colab mounts Drive in the cloud.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
SHARED = Path('/content/drive/MyDrive/Cadenza Challenge')   # the shortcut name must match
assert SHARED.exists(), 'Add a shortcut to the shared folder in My Drive first (see top of notebook)'
print('shared folder ok:', SHARED)
print(sorted(p.name for p in SHARED.iterdir()))

In [ ]:
import hashlib, shutil

FILES = {   # name -> md5 (same values as notebooks 02 and 03)
    'metadata.zip': '7ff49002ae62023cd547ff057cbba461',
    'EnsembleSet_Mix_1.zip': '5bdcd0154da69e7516aef2d3062a47f0',
    'CadenzaWoodwind.zip': 'cdc08be8482662c5023b5d4060d1b064',
    'cadenza_cad2_task2_eval.v1_1.tar.gz': '54268aa8dc08035cd012483bba701d09',
}

def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

missing = [n for n in FILES if not (SHARED / n).exists()]
assert not missing, f'Not found directly in {SHARED}: {missing} (are they in a subfolder?)'
print('all files found')

## Part A - copy from Drive and build the dataset (everyone)
Takes a few minutes. Copies skip files that are already local and verified.

In [ ]:
dl = Path('/content/zenodo_download'); dl.mkdir(exist_ok=True)
for name, want in FILES.items():
    if name.endswith('.tar.gz'):
        continue                      # eval set is handled in Part B
    dst = dl / name
    if not (dst.exists() and md5sum(dst) == want):
        print('copying', name)
        shutil.copy(SHARED / name, dst)
    assert md5sum(dst) == want, f'md5 mismatch for {name} - re-upload it to the shared folder'
    print('OK', name)

In [ ]:
root = Path('/content/cadenza_data/cad2/task2')
clarity_dir = Path('/content/clarity')
!PYTHONPATH={clarity_dir} python {clarity_dir}/recipes/cad2/task2/process_dataset/process_zenodo_download.py path.zenodo_download_path={dl} path.root={root}

In [ ]:
meta = root / 'metadata'
for name in ['scenes.valid.json', 'scene_listeners.valid.json', 'music.valid.json', 'listeners.valid.json']:
    assert (meta / name).exists(), f'missing {name}'
print('dataset ready:', root)

## Part B - eval set (optional, used by notebook 02)
Copied from the shared Drive folder and extracted on Colab's local disk.

In [ ]:
EV = 'cadenza_cad2_task2_eval.v1_1.tar.gz'
dst = Path('/content') / EV
if not (dst.exists() and md5sum(dst) == FILES[EV]):
    shutil.copy(SHARED / EV, dst)
assert md5sum(dst) == FILES[EV], 'eval md5 mismatch - re-upload it to the shared folder'
!cd /content && tar -xzf {EV} && ls cadenza_data/cad2/task2